# API call code for theoretical and operationalized SEPARATED prompt variants, on GPT-5. 
# We separate by 1. Title + Excerpt Frames 2. Binary tasks (demands present + quoted voices) 3. thematic present

In [16]:
import pandas as pd
df = pd.read_excel("frame_gold.xlsx")
df.head()

,uuid,title,protest_excerpt,title_frame,excerpt_frame,demands_present,authority_quote,protester_quote,thematic_present
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,Third politician in a month killed in Ecuador,The gang boss was moved out of a jail with lig...,residual,grievance,yes,no,no,no
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,Amazon CEO tells staff to work in office three...,This comes after Amazon said it would lay off ...,residual,widespread_support,yes,no,yes,yes
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,Gaza ceasefire protest in Capitol rotunda lead...,Capitol rotunda on Sept. 27. Photo: Michael Ro...,grievance,disruption,yes,yes,no,yes
3,155faae5-c8b6-441e-a20e-dce06e01cf32,"Daniel Penny, accused in Jordan Neely subway c...",Neely and could not have foreseen his untimely...,residual,grievance,yes,yes,yes,no
4,50fea669-6828-44de-8a70-b448e398be11,Legal scholars raising constitutional argument...,Not all in the legal community agree – and wha...,residual,protester_violence,no,yes,no,yes


# load prompts, df

In [19]:
# If needed:
# %pip install openai pandas openpyxl

import os
import json
import time
import pandas as pd

from openai import OpenAI

import frame_prompts_separated
import importlib
importlib.reload(frame_prompts_separated)

from frame_prompts_separated import (
    THEORETICAL_FRAMES_PROMPT,
    THEORETICAL_BINARY_PROMPT,
    THEORETICAL_THEMATIC_PROMPT,
)

# -----------------------------
# MODEL SETTINGS
# -----------------------------

MODEL = "gpt-5-mini"
REASONING_EFFORT = "medium"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# -----------------------------
# LOAD GOLD DATA
# -----------------------------

df = pd.read_excel("frame_gold.xlsx")

required_columns = [
    "uuid",
    "title",
    "protest_excerpt",
    "title_frame",
    "excerpt_frame",
    "demands_present",
    "protester_quote",
    "authority_quote",
    "thematic_present",
]

missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required_columns].copy()

df["title"] = df["title"].fillna("").astype(str)
df["protest_excerpt"] = df["protest_excerpt"].fillna("").astype(str)


# Normalize gold labels
label_columns = [
    "title_frame",
    "excerpt_frame",
    "demands_present",
    "protester_quote",
    "authority_quote",
    "thematic_present",
]

for col in label_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

# In case your source file still uses "violence"
df["title_frame"] = df["title_frame"].replace(
    "violence", "protester_violence"
)

df["excerpt_frame"] = df["excerpt_frame"].replace(
    "violence", "protester_violence"
)


# -----------------------------
# TEST FIRST
# -----------------------------

test_df = df.head(3).copy()

# Change this ONE line when ready for all 120:
RUN_DF = df

# For full validation:
# RUN_DF = df

print(f"Running {len(RUN_DF)} samples.")

Running 120 samples.


# schemas and API helper

In [20]:
FRAME_LABELS = [
    "protester_violence",
    "harm_to_protesters",
    "property_damage",
    "disruption",
    "grievance",
    "widespread_support",
    "peaceful",
    "impactful",
    "residual",
]


# ============================================================
# STRUCTURED OUTPUT SCHEMAS
# ============================================================

FRAMES_SCHEMA = {
    "type": "object",
    "properties": {
        "title_frame": {
            "type": "string",
            "enum": FRAME_LABELS,
        },
        "excerpt_frame": {
            "type": "string",
            "enum": FRAME_LABELS,
        },
    },
    "required": [
        "title_frame",
        "excerpt_frame",
    ],
    "additionalProperties": False,
}


BINARY_SCHEMA = {
    "type": "object",
    "properties": {
        "demands_present": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "protester_quote": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "authority_quote": {
            "type": "string",
            "enum": ["yes", "no"],
        },
    },
    "required": [
        "demands_present",
        "protester_quote",
        "authority_quote",
    ],
    "additionalProperties": False,
}


THEMATIC_SCHEMA = {
    "type": "object",
    "properties": {
        "thematic_present": {
            "type": "string",
            "enum": ["yes", "no"],
        }
    },
    "required": ["thematic_present"],
    "additionalProperties": False,
}


# ============================================================
# FORMAT ARTICLE INPUT
# ============================================================

def format_input(title, excerpt):
    return f"""TITLE:
{title}

EXCERPT:
{excerpt}
"""


# ============================================================
# GENERIC API CALL
# ============================================================

def call_classifier(
    title,
    excerpt,
    system_prompt,
    schema,
    schema_name,
    max_retries=3
):

    user_input = format_input(title, excerpt)

    for attempt in range(max_retries):

        try:
            response = client.responses.create(
                model=MODEL,
                instructions=system_prompt,
                input=user_input,

                reasoning={
                    "effort": REASONING_EFFORT
                },

                text={
                    "verbosity": "low",
                    "format": {
                        "type": "json_schema",
                        "name": schema_name,
                        "strict": True,
                        "schema": schema,
                    },
                },
            )

            parsed = json.loads(response.output_text)

            # Token details
            cached_tokens = None
            reasoning_tokens = None

            try:
                cached_tokens = (
                    response.usage
                    .input_tokens_details
                    .cached_tokens
                )
            except Exception:
                pass

            try:
                reasoning_tokens = (
                    response.usage
                    .output_tokens_details
                    .reasoning_tokens
                )
            except Exception:
                pass

            return {
                **parsed,
                "response_id": response.id,
                "input_tokens": response.usage.input_tokens,
                "cached_tokens": cached_tokens,
                "output_tokens": response.usage.output_tokens,
                "reasoning_tokens": reasoning_tokens,
                "error": None,
            }

        except Exception as e:

            print(
                f"Attempt {attempt + 1}/{max_retries} failed: "
                f"{type(e).__name__}: {e}"
            )

            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)

            else:
                return {
                    "response_id": None,
                    "input_tokens": None,
                    "cached_tokens": None,
                    "output_tokens": None,
                    "reasoning_tokens": None,
                    "error": str(e),
                }

# run all three tasks, save immediately, and merge by uuid into one final df

In [21]:
# ============================================================
# 1. FRAMES
# ============================================================

frame_results = []

print("\n==============================")
print("RUNNING FRAME CLASSIFICATION")
print("==============================")

for count, (_, row) in enumerate(RUN_DF.iterrows(), start=1):

    print(f"[{count}/{len(RUN_DF)}] {row['uuid']}")

    result = call_classifier(
        title=row["title"],
        excerpt=row["protest_excerpt"],
        system_prompt=THEORETICAL_FRAMES_PROMPT,
        schema=FRAMES_SCHEMA,
        schema_name="theoretical_frames",
    )

    result["uuid"] = row["uuid"]
    frame_results.append(result)


frames_df = pd.DataFrame(frame_results)

frames_df = frames_df.rename(columns={
    "title_frame": "gpt_title_frame",
    "excerpt_frame": "gpt_excerpt_frame",
    "response_id": "frames_response_id",
    "input_tokens": "frames_input_tokens",
    "cached_tokens": "frames_cached_tokens",
    "output_tokens": "frames_output_tokens",
    "reasoning_tokens": "frames_reasoning_tokens",
    "error": "frames_error",
})

frames_df.to_excel(
    "theoretical_separated_frames_raw.xlsx",
    index=False
)

print("Frames saved.")


# ============================================================
# 2. BINARY VARIABLES
# ============================================================

binary_results = []

print("\n==============================")
print("RUNNING BINARY CLASSIFICATION")
print("==============================")

for count, (_, row) in enumerate(RUN_DF.iterrows(), start=1):

    print(f"[{count}/{len(RUN_DF)}] {row['uuid']}")

    result = call_classifier(
        title=row["title"],
        excerpt=row["protest_excerpt"],
        system_prompt=THEORETICAL_BINARY_PROMPT,
        schema=BINARY_SCHEMA,
        schema_name="theoretical_binary",
    )

    result["uuid"] = row["uuid"]
    binary_results.append(result)


binary_df = pd.DataFrame(binary_results)

binary_df = binary_df.rename(columns={
    "demands_present": "gpt_demands_present",
    "protester_quote": "gpt_protester_quote",
    "authority_quote": "gpt_authority_quote",
    "response_id": "binary_response_id",
    "input_tokens": "binary_input_tokens",
    "cached_tokens": "binary_cached_tokens",
    "output_tokens": "binary_output_tokens",
    "reasoning_tokens": "binary_reasoning_tokens",
    "error": "binary_error",
})

binary_df.to_excel(
    "theoretical_separated_binary_raw.xlsx",
    index=False
)

print("Binary results saved.")


# ============================================================
# 3. THEMATIC
# ============================================================

thematic_results = []

print("\n==============================")
print("RUNNING THEMATIC CLASSIFICATION")
print("==============================")

for count, (_, row) in enumerate(RUN_DF.iterrows(), start=1):

    print(f"[{count}/{len(RUN_DF)}] {row['uuid']}")

    result = call_classifier(
        title=row["title"],
        excerpt=row["protest_excerpt"],
        system_prompt=THEORETICAL_THEMATIC_PROMPT,
        schema=THEMATIC_SCHEMA,
        schema_name="theoretical_thematic",
    )

    result["uuid"] = row["uuid"]
    thematic_results.append(result)


thematic_df = pd.DataFrame(thematic_results)

thematic_df = thematic_df.rename(columns={
    "thematic_present": "gpt_thematic_present",
    "response_id": "thematic_response_id",
    "input_tokens": "thematic_input_tokens",
    "cached_tokens": "thematic_cached_tokens",
    "output_tokens": "thematic_output_tokens",
    "reasoning_tokens": "thematic_reasoning_tokens",
    "error": "thematic_error",
})

thematic_df.to_excel(
    "theoretical_separated_thematic_raw.xlsx",
    index=False
)

print("Thematic results saved.")


# ============================================================
# 4. MERGE EVERYTHING BACK ON UUID
# ============================================================

results_df = (
    RUN_DF
    .merge(
        frames_df,
        on="uuid",
        how="left",
        validate="one_to_one"
    )
    .merge(
        binary_df,
        on="uuid",
        how="left",
        validate="one_to_one"
    )
    .merge(
        thematic_df,
        on="uuid",
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 5. TOTAL TOKEN USAGE PER ARTICLE
# ============================================================

results_df["total_input_tokens"] = (
    results_df["frames_input_tokens"].fillna(0)
    + results_df["binary_input_tokens"].fillna(0)
    + results_df["thematic_input_tokens"].fillna(0)
)

results_df["total_output_tokens"] = (
    results_df["frames_output_tokens"].fillna(0)
    + results_df["binary_output_tokens"].fillna(0)
    + results_df["thematic_output_tokens"].fillna(0)
)

results_df["total_reasoning_tokens"] = (
    results_df["frames_reasoning_tokens"].fillna(0)
    + results_df["binary_reasoning_tokens"].fillna(0)
    + results_df["thematic_reasoning_tokens"].fillna(0)
)

results_df["total_cached_tokens"] = (
    results_df["frames_cached_tokens"].fillna(0)
    + results_df["binary_cached_tokens"].fillna(0)
    + results_df["thematic_cached_tokens"].fillna(0)
)


# ============================================================
# 6. SAVE MERGED VALIDATION FILE
# ============================================================

output_file = (
    f"merged_theoretical_separated_"
    f"{MODEL}_{REASONING_EFFORT}.xlsx"
)

results_df.to_excel(
    output_file,
    index=False
)

print(f"\nMerged results saved to: {output_file}")


# ============================================================
# 7. QUICK TOKEN SUMMARY
# ============================================================

print("\nAverage tokens per ARTICLE across all 3 calls:")

print(
    f"Input:     "
    f"{results_df['total_input_tokens'].mean():,.0f}"
)

print(
    f"Cached:    "
    f"{results_df['total_cached_tokens'].mean():,.0f}"
)

print(
    f"Output:    "
    f"{results_df['total_output_tokens'].mean():,.0f}"
)

print(
    f"Reasoning: "
    f"{results_df['total_reasoning_tokens'].mean():,.0f}"
)

results_df.head()


RUNNING FRAME CLASSIFICATION
[1/120] d1334539-4cf3-4c5f-a0ef-88db5f167395
[2/120] b088e93f-1de4-4021-97bf-a90700b0a0ac
[3/120] fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad
[4/120] 155faae5-c8b6-441e-a20e-dce06e01cf32
[5/120] 50fea669-6828-44de-8a70-b448e398be11
[6/120] 81efb265-e3ba-4508-adc7-73cbc029f359
[7/120] 5fb219a8-9df7-4ad0-bc57-d46be3e327e0
[8/120] 252fb33d-5e6c-4cff-acc0-98c253f6c4b8
[9/120] 4acf5304-e74d-4a4f-bee0-caf65bd02abe
[10/120] 74fd09e2-35a3-495b-9b59-1610ec3398e1
[11/120] 7d9d161f-8087-4e0b-89e2-4a9bc1dcfd9d
[12/120] 0c5fa89f-f32f-4014-98e5-e8ad481b8a2f
[13/120] b2a2ef91-319a-47bf-9a61-1f8b4e192abf
[14/120] a92ef5b3-9891-4492-8a32-5024d7cc52dc
[15/120] c13350d6-ae7e-43f9-8999-17e530a4c80b
[16/120] 874c16ec-4684-40c7-8928-06baaa20260d
[17/120] 0192cd22-a165-4eb8-bf6d-da6165164b2e
[18/120] 95c2c809-f17e-4d9d-af74-26ba987929cd
[19/120] 109cd735-179b-4d53-ba32-ac1b70d5d990
[20/120] 36ab3319-2965-4075-80bd-14627327ab65
[21/120] f4bf7615-7d65-4e3d-b332-fbf590087f99
[22/120] 58a5

,uuid,title,protest_excerpt,title_frame,excerpt_frame,demands_present,protester_quote,authority_quote,thematic_present,gpt_title_frame,...,thematic_response_id,thematic_input_tokens,thematic_cached_tokens,thematic_output_tokens,thematic_reasoning_tokens,thematic_error,total_input_tokens,total_output_tokens,total_reasoning_tokens,total_cached_tokens
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,Third politician in a month killed in Ecuador,The gang boss was moved out of a jail with lig...,residual,grievance,yes,no,no,no,residual,...,resp_0bfbc9519a22523a006a8f5476e11887d1a309042...,1106,0,114,64,None,3817,852,704,0
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,Amazon CEO tells staff to work in office three...,This comes after Amazon said it would lay off ...,residual,widespread_support,yes,yes,no,yes,residual,...,resp_0562c412b0cde001006a8f5478db0087d1a8c2b29...,1172,0,299,256,None,4015,990,832,0
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,Gaza ceasefire protest in Capitol rotunda lead...,Capitol rotunda on Sept. 27. Photo: Michael Ro...,grievance,disruption,yes,no,yes,yes,disruption,...,resp_0500ce11677927ed006a8f547cb8bc87d1b0aa11e...,1164,0,341,320,None,3991,1016,832,0
3,155faae5-c8b6-441e-a20e-dce06e01cf32,"Daniel Penny, accused in Jordan Neely subway c...",Neely and could not have foreseen his untimely...,residual,grievance,yes,yes,yes,no,residual,...,resp_00483022c61d9a88006a8f5482c77087d1bd0bdd0...,1108,0,121,64,None,3823,740,576,0
4,50fea669-6828-44de-8a70-b448e398be11,Legal scholars raising constitutional argument...,Not all in the legal community agree – and wha...,residual,protester_violence,no,no,yes,yes,residual,...,resp_097b8d27fc88496e006a8f5484e74887d19b7bf59...,1406,0,186,128,None,4717,1255,1088,0


# results summary table

In [22]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report,
    confusion_matrix
)
import simpledorff
import numpy as np

# --------------------------------------------------
# EVALUATION FUNCTIONS
# --------------------------------------------------

TASKS = {
    "title_frame": {
        "gold": "title_frame",
        "pred": "gpt_title_frame",
        "type": "multiclass",
    },
    "excerpt_frame": {
        "gold": "excerpt_frame",
        "pred": "gpt_excerpt_frame",
        "type": "multiclass",
    },
    "demands_present": {
        "gold": "demands_present",
        "pred": "gpt_demands_present",
        "type": "binary",
    },
    "protester_quote": {
        "gold": "protester_quote",
        "pred": "gpt_protester_quote",
        "type": "binary",
    },
    "authority_quote": {
        "gold": "authority_quote",
        "pred": "gpt_authority_quote",
        "type": "binary",
    },
    "thematic_present": {
        "gold": "thematic_present",
        "pred": "gpt_thematic_present",
        "type": "binary",
    },
}


def kripp_alpha_two_annotators(gold, pred):
    """
    Krippendorff's alpha for nominal labels,
    treating Human Gold and GPT as two annotators.
    """

    tmp = pd.DataFrame({
        "item_id": np.arange(len(gold)).repeat(2),
        "annotator_id": ["human", "gpt"] * len(gold),
        "label": np.column_stack([gold, pred]).reshape(-1),
    })

    return simpledorff.calculate_krippendorffs_alpha_for_df(
        tmp,
        experiment_col="item_id",
        annotator_col="annotator_id",
        class_col="label",
    )


def evaluate_task(data, gold_col, pred_col, task_type):

    valid = data[[gold_col, pred_col]].dropna()

    y_true = valid[gold_col].astype(str)
    y_pred = valid[pred_col].astype(str)

    result = {
        "n": len(valid),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "kripp_alpha": kripp_alpha_two_annotators(
            y_true.tolist(),
            y_pred.tolist(),
        ),
    }

    if task_type == "binary":

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=["yes"],
            average=None,
            zero_division=0,
        )

        result["yes_precision"] = precision[0]
        result["yes_recall"] = recall[0]
        result["yes_f1"] = f1[0]

    else:

        result["weighted_f1"] = f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        )

    return result


# --------------------------------------------------
# SUMMARY TABLE
# --------------------------------------------------

summary_rows = []

for task_name, config in TASKS.items():

    metrics = evaluate_task(
        results_df,
        gold_col=config["gold"],
        pred_col=config["pred"],
        task_type=config["type"],
    )

    summary_rows.append({
        "task": task_name,
        **metrics,
    })


summary_df = pd.DataFrame(summary_rows)

# Arrange the most useful columns first.
preferred_cols = [
    "task",
    "n",
    "accuracy",
    "macro_f1",
    "yes_precision",
    "yes_recall",
    "yes_f1",
    "weighted_f1",
    "kripp_alpha",
]

summary_df = summary_df[
    [c for c in preferred_cols if c in summary_df.columns]
]

print("\n==============================")
print("VALIDATION SUMMARY")
print("==============================\n")

display(
    summary_df.round(3)
)


# --------------------------------------------------
# DETAILED MULTICLASS REPORTS
# --------------------------------------------------

for task in ["title_frame", "excerpt_frame"]:

    gold_col = TASKS[task]["gold"]
    pred_col = TASKS[task]["pred"]

    valid = results_df[[gold_col, pred_col]].dropna()

    print("\n")
    print("=" * 70)
    print(task.upper())
    print("=" * 70)

    print(
        classification_report(
            valid[gold_col],
            valid[pred_col],
            labels=FRAME_LABELS,
            zero_division=0,
        )
    )

    cm = pd.DataFrame(
        confusion_matrix(
            valid[gold_col],
            valid[pred_col],
            labels=FRAME_LABELS,
        ),
        index=[f"gold_{x}" for x in FRAME_LABELS],
        columns=[f"gpt_{x}" for x in FRAME_LABELS],
    )

    display(cm)


# --------------------------------------------------
# SAVE summary
# --------------------------------------------------
summary_df.to_excel(
    f"summary_theoretical_separated_gpt-5-mini.xlsx",
    index=False
)


VALIDATION SUMMARY



,task,n,accuracy,macro_f1,yes_precision,yes_recall,yes_f1,weighted_f1,kripp_alpha
0,title_frame,120,0.575,0.476,NaN,NaN,NaN,0.620,0.412
1,excerpt_frame,120,0.617,0.572,NaN,NaN,NaN,0.598,0.534
2,demands_present,120,0.783,0.689,0.777,0.964,0.860,NaN,0.381
3,protester_quote,120,0.817,0.816,0.730,0.902,0.807,NaN,0.634
4,authority_quote,120,0.817,0.791,0.805,0.933,0.864,NaN,0.584
5,thematic_present,120,0.675,0.670,0.578,0.923,0.711,NaN,0.342




TITLE_FRAME
                    precision    recall  f1-score   support

protester_violence       0.29      0.80      0.42         5
harm_to_protesters       0.35      1.00      0.52         6
   property_damage       0.50      1.00      0.67         1
        disruption       0.21      0.75      0.33         4
         grievance       0.77      0.71      0.74        14
widespread_support       0.62      0.83      0.71         6
          peaceful       0.00      0.00      0.00         0
         impactful       0.17      0.40      0.24         5
          residual       1.00      0.48      0.65        79

          accuracy                           0.57       120
         macro avg       0.43      0.66      0.48       120
      weighted avg       0.83      0.57      0.62       120



,gpt_protester_violence,gpt_harm_to_protesters,gpt_property_damage,gpt_disruption,gpt_grievance,gpt_widespread_support,gpt_peaceful,gpt_impactful,gpt_residual
gold_protester_violence,4,0,1,0,0,0,0,0,0
gold_harm_to_protesters,0,6,0,0,0,0,0,0,0
gold_property_damage,0,0,1,0,0,0,0,0,0
gold_disruption,0,0,0,3,0,0,0,1,0
gold_grievance,0,0,0,3,10,0,1,0,0
gold_widespread_support,0,0,0,0,1,5,0,0,0
gold_peaceful,0,0,0,0,0,0,0,0,0
gold_impactful,1,0,0,2,0,0,0,2,0
gold_residual,9,11,0,6,2,3,1,9,38




EXCERPT_FRAME
                    precision    recall  f1-score   support

protester_violence       0.69      0.81      0.75        31
harm_to_protesters       0.48      0.83      0.61        12
   property_damage       1.00      0.50      0.67         4
        disruption       0.46      0.35      0.40        17
         grievance       0.66      0.84      0.74        25
widespread_support       0.80      0.29      0.42        14
          peaceful       1.00      0.67      0.80         3
         impactful       0.29      0.25      0.27         8
          residual       1.00      0.33      0.50         6

          accuracy                           0.62       120
         macro avg       0.71      0.54      0.57       120
      weighted avg       0.65      0.62      0.60       120



,gpt_protester_violence,gpt_harm_to_protesters,gpt_property_damage,gpt_disruption,gpt_grievance,gpt_widespread_support,gpt_peaceful,gpt_impactful,gpt_residual
gold_protester_violence,25,1,0,0,1,1,0,3,0
gold_harm_to_protesters,2,10,0,0,0,0,0,0,0
gold_property_damage,2,0,2,0,0,0,0,0,0
gold_disruption,6,1,0,6,2,0,0,2,0
gold_grievance,0,3,0,1,21,0,0,0,0
gold_widespread_support,0,2,0,0,8,4,0,0,0
gold_peaceful,0,1,0,0,0,0,2,0,0
gold_impactful,0,1,0,5,0,0,0,2,0
gold_residual,1,2,0,1,0,0,0,0,2


In [15]:
import pandas as pd

summary_old = pd.read_excel(
    "summary_theoretical_separated.xlsx"
)

display(summary_old)

,task,n,accuracy,macro_f1,yes_precision,yes_recall,yes_f1,weighted_f1,kripp_alpha
0,title_frame,120,0.758333,0.593236,NaN,NaN,NaN,0.769434,0.587440
1,excerpt_frame,120,0.683333,0.617820,NaN,NaN,NaN,0.674457,0.621126
2,demands_present,120,0.833333,0.791014,0.846154,0.927711,0.885057,NaN,0.583769
3,protester_quote,120,0.850000,0.847285,0.811321,0.843137,0.826923,NaN,0.695843
4,authority_quote,120,0.750000,0.686520,0.727273,0.960000,0.827586,NaN,0.375653
5,thematic_present,120,0.775000,0.773727,0.711864,0.807692,0.756757,NaN,0.549340
